# PDCP · N05 — E1 main experiment (+ E1b Mooney-style pairs)
**Notebook 5 of 13.** Kaggle: **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 4–7 h (stops itself before 11.5 h).

**Inputs:** N03 output (`pairs.parquet`, `drawings.parquet`, `images/768/`). Optional: a previous N05 output to **resume** (set `RESUME_ROOT`).

**Decisions from the N04 pilot (18 Nov 2026)**
- Models: **Q3VL-2B, Q3VL-4B, Q25VL-7B (4-bit), InternVL-4B** (+ Gemma3-4B if `HF_TOKEN` is set). Dropped: SmolVLM2 (8 % parseable, answers only "1"), Q3VL-8B 4-bit (≈ 55 s/query on T4).
- Every pair in **both image orders** (pilot showed 65–91 % preference for one image position).
- Reasons shortened to ≤ 25 words, `max_new_tokens` 96, batched inference, InternVL with one 448-px tile per image.
- Worker v0.3: deadline-aware and resumable — if the session ends early, attach this notebook's output as `RESUME_ROOT` and run again; finished queries are skipped.

**Queries per model:** 1,926 main pairs × 2 orders + 289 E1b pairs × 2 orders = **4,430**.

**Outputs** (`/kaggle/working/n05_output/`): `results/e1/<model>.jsonl`, `e1_queries.csv`, `e1_summary.csv`, `e1_status.json`, `pdcp_infer.py` v0.3, logs, figures, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"   # searched for pairs.parquet
RESUME_ROOT = None          # e.g. "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output" to continue a previous run
OUTPUT_DIR = "/kaggle/working/n05_output"
MODELS = ["Q25VL-7B", "Q3VL-4B", "InternVL-4B", "Q3VL-2B", "Gemma3-4B"]   # longest first (balanced over 2 GPUs)
PROMPT = "P-Def"
MAX_NEW_TOKENS = 96
SESSION_HOURS = 11.3        # workers stop cleanly after this (Kaggle hard limit 12 h)
SEED = 2026

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
os.makedirs(f"{OUTPUT_DIR}/results/e1", exist_ok=True); os.makedirs(f"{OUTPUT_DIR}/logs", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
N_GPU = torch.cuda.device_count(); print("GPUs:", N_GPU); assert N_GPU >= 1, "Set Accelerator = GPU T4 x2"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes",
                    "qwen-vl-utils", "num2words"], capture_output=True, text=True)
print(subprocess.run([sys.executable, "-c", "import transformers, bitsandbytes; print('transformers', transformers.__version__, '| bnb', bitsandbytes.__version__)"],
                     capture_output=True, text=True).stdout)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN found")
except Exception as e:
    print("No HF_TOKEN secret -> Gemma skipped:", type(e).__name__)
if HF_TOKEN is None:
    MODELS = [m for m in MODELS if m != "Gemma3-4B"]
print("models:", MODELS)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

GPUs: 2
transformers 5.23.0 | bnb 0.50.2

No HF_TOKEN secret -> Gemma skipped: BackendError
models: ['Q25VL-7B', 'Q3VL-4B', 'InternVL-4B', 'Q3VL-2B']


## 1. Query list (main + E1b, both orders, shuffled so a partial run is still balanced)

In [3]:
import numpy as np, pandas as pd
pp = sorted(glob.glob(f"{N03_ROOT}/**/pairs.parquet", recursive=True)); assert pp, "attach the N03 output dataset"
N03 = os.path.dirname(pp[0])
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
png = dict(zip(DR.drawing_id, DR.png))
E = PR[PR.set.isin(["main", "e1b_mooney"])]
rows = []
for r in E.itertuples():
    a, b = f"{N03}/{png[r.drawing_A]}", f"{N03}/{png[r.drawing_B]}"
    for order in ("AB", "BA"):
        img1, img2 = (a, b) if order == "AB" else (b, a)
        corr = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
        rows.append({"pair_id": r.pair_id, "order": order, "img1": img1, "img2": img2, "correct_img": corr, "set": r.set,
                     "dstress": r.dstress_design, "cond": r.cond, "family": f"{r.family}{r.n}"})
Q = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
assert all(os.path.exists(p) for p in Q.img1.head(50))
Q.to_csv(f"{OUTPUT_DIR}/e1_queries.csv", index=False)
print(len(Q), "queries per model |", Q.set.value_counts().to_dict(), "| correct:", Q.correct_img.value_counts().to_dict())
if RESUME_ROOT:
    for f in glob.glob(f"{RESUME_ROOT}/**/results/e1/*.jsonl", recursive=True):
        shutil.copy(f, f"{OUTPUT_DIR}/results/e1/{os.path.basename(f)}"); print("resumed", os.path.basename(f), sum(1 for _ in open(f)), "rows")

4430 queries per model | {'main': 3852, 'e1b_mooney': 578} | correct: {'1': 1857, '2': 1857, 'same': 716}


## 2. Inference worker v0.3

In [4]:
%%writefile /kaggle/working/n05_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.3.0). One model per process.
v0.3: processor_kwargs API, per-model processor options + default batch, 25-word reasons, early abort on unparseable output.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {   # batch = default batch size on one T4 (from the N04 pilot); proc = extra processor options
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None, "batch": 8, "proc": {}},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4", "batch": 2, "proc": {}},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4", "batch": 4, "proc": {}},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None, "batch": 4, "proc": {}},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4,
                    "proc": {"crop_to_patches": False}},   # one 448-px tile per image (pilot: tiling -> 11 s/query, OOM at batch 4)
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None, "batch": 1, "proc": {}},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 25 words>"}')
PROMPTS = {
    "P-Def": DEFINITION + "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n" + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    return [{"role": "user", "content": [
        {"type": "text", "text": SYSTEM + "\n\nImage 1:"}, im(img1),
        {"type": "text", "text": "Image 2:"}, im(img2),
        {"type": "text", "text": PROMPTS[prompt_id]}]}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    pk = dict(MODELS[self.key].get("proc", {}))
                    if len(batch) > 1:
                        pk["padding"] = True
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        processor_kwargs=pk)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    imgs = [[a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ab]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True, **MODELS[self.key].get("proc", {}))
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=0, help="0 = model default")
    ap.add_argument("--min_parse", type=float, default=0.5, help="abort if parse rate after 40 queries is below this")
    ap.add_argument("--deadline", type=float, default=0, help="unix time after which the worker stops cleanly")
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], a.prompt) not in done]
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], a.prompt, a.max_new)
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new)[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = a.batch if a.batch > 0 else MODELS[a.model].get("batch", 1)
        summ["batch"] = bs
        for i in range(0, len(todo), bs):
            chunk = todo[i:i + bs]
            if a.deadline and time.time() > a.deadline:
                log("deadline reached - stopping cleanly (resumable)"); summ["stopped_at_deadline"] = True; break
            t0 = time.time()
            try:
                texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], a.prompt, a.max_new)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    log("OOM at batch %d -> falling back to batch 1" % bs); bs = 1
                    texts, n_in = [], 0
                    for r in chunk:
                        t, n_in = R.generate([(load(r["img1"]), load(r["img2"]))], a.prompt, a.max_new); texts += t
                else:
                    raise
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": a.prompt, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n >= 40 and n - len(chunk) < 40 and n_ok / n < a.min_parse:
                log(f"parse rate {n_ok/n:.2f} < {a.min_parse} after {n} queries - aborting this model")
                summ["aborted_low_parse"] = True; break
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n05_output/pdcp_infer.py


## 3. Run (two model processes in parallel, one per GPU)

In [5]:
HF_CACHE = "/tmp/hf_cache"
sys.path.insert(0, OUTPUT_DIR); import pdcp_infer as I; importlib.reload(I)
DEADLINE = T0 + SESSION_HOURS * 3600
queue, running = list(MODELS), {}
def launch(model, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HUB_CACHE=HF_CACHE, TOKENIZERS_PARALLELISM="false",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/e1_queries.csv",
           "--out", f"{OUTPUT_DIR}/results/e1/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/e1/{model}.summary.json",
           "--prompt", PROMPT, "--max_new", str(MAX_NEW_TOKENS), "--deadline", str(DEADLINE)]
    if HF_TOKEN: cmd += ["--hf_token", HF_TOKEN]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "a")
    running[gpu] = (model, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), time.time(), logf)
    print(f"[{(time.time()-T0)/60:6.1f} min] start {model} on GPU {gpu}")
last_report = 0
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue and time.time() < DEADLINE - 1800:
            launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(30)
    if time.time() - last_report > 1800:      # progress every 30 min
        last_report = time.time()
        for m in MODELS:
            f = f"{OUTPUT_DIR}/results/e1/{m}.jsonl"
            if os.path.exists(f): print(f"   {m}: {sum(1 for _ in open(f))}/{len(Q)}")
    for gpu, (model, p, st, logf) in list(running.items()):
        if p.poll() is not None:
            logf.close(); del running[gpu]
            tail = open(f"{OUTPUT_DIR}/logs/{model}.log").read().strip().splitlines()[-1:]
            print(f"[{(time.time()-T0)/60:6.1f} min] end {model}: {' '.join(tail)[-300:]}")
            for mid in I.MODELS[model]["ids"]: shutil.rmtree(f"{HF_CACHE}/models--{mid.replace('/', '--')}", ignore_errors=True)
print(f"run phase finished after {(time.time()-T0)/3600:.2f} h")

[   0.8 min] start Q25VL-7B on GPU 0
[   0.8 min] start Q3VL-4B on GPU 1
   Q25VL-7B: 268/4430
   Q3VL-4B: 460/4430
   Q25VL-7B: 572/4430
   Q3VL-4B: 972/4430
   Q25VL-7B: 876/4430
   Q3VL-4B: 1492/4430
   Q25VL-7B: 1180/4430
   Q3VL-4B: 2016/4430
   Q25VL-7B: 1484/4430
   Q3VL-4B: 2544/4430
   Q25VL-7B: 1788/4430
   Q3VL-4B: 3072/4430
   Q25VL-7B: 2096/4430
   Q3VL-4B: 3604/4430
   Q25VL-7B: 2400/4430
   Q3VL-4B: 4140/4430
[ 257.8 min] end Q3VL-4B: [Q3VL-4B 01:16:00] done: {"ok": true, "n": 4430, "sec_per_query": 3.435, "parse_rate": 1.0, "acc_on_parsed": 0.272, "vram_peak_gb": 12.76}
[ 257.8 min] start InternVL-4B on GPU 1
   Q25VL-7B: 2704/4430
   Q3VL-4B: 4430/4430
   InternVL-4B: 412/4430
   Q25VL-7B: 3012/4430
   Q3VL-4B: 4430/4430
   InternVL-4B: 1416/4430
   Q25VL-7B: 3316/4430
   Q3VL-4B: 4430/4430
   InternVL-4B: 2412/4430
   Q25VL-7B: 3624/4430
   Q3VL-4B: 4430/4430
   InternVL-4B: 3424/4430
   Q25VL-7B: 3928/4430
   Q3VL-4B: 4430/4430
   InternVL-4B: 4424/4430
[ 391.8 min] 

## 4. Status and first look at results

In [6]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
R = []
for m in MODELS:
    f = f"{OUTPUT_DIR}/results/e1/{m}.jsonl"
    if os.path.exists(f): R += [json.loads(l) for l in open(f)]
R = pd.DataFrame(R)
status = {"queries_per_model": len(Q), "hours": round((time.time() - T0) / 3600, 2), "models": {}}
if len(R):
    R = R.merge(Q[["pair_id", "order", "set", "dstress", "cond", "family"]], on=["pair_id", "order"], how="left")
    R["ans"] = R.parsed.map(lambda p: p.get("answer"))
    R["conf"] = R.parsed.map(lambda p: p.get("confidence"))
    rows = []
    for m, g in R.groupby("model"):
        mn = g[g.set == "main"]; ok = mn[mn.parse_ok]; nz = ok[ok.dstress > 0]
        # order-averaged pair accuracy: a pair counts 0, 0.5 or 1 depending on how many of its two orders were right
        pa = nz.groupby("pair_id").correct.mean()
        rows.append({"model": m, "done": len(g), "complete": len(g) >= len(Q), "parse_rate": round(g.parse_ok.mean(), 3),
                     "acc_3afc_main": round(ok.correct.mean(), 3), "acc_dstress>0": round(nz.correct.mean(), 3),
                     "pair_acc_both_orders": round(pa.mean(), 3), "p_answer_1": round((ok.ans == "1").mean(), 3),
                     "p_answer_same": round((ok.ans == "same").mean(), 3),
                     "acc_order_AB": round(nz[nz.order == "AB"].correct.mean(), 3), "acc_order_BA": round(nz[nz.order == "BA"].correct.mean(), 3),
                     **{f"acc_{c}": round(nz[nz.cond == c].correct.mean(), 3) for c in ["congruent", "neutral", "conflict_nu", "conflict_eld", "conflict_xr", "conflict_all"]},
                     "acc_e1b": round(g[(g.set == "e1b_mooney") & g.parse_ok].correct.mean(), 3),
                     "mean_conf": round(pd.to_numeric(ok.conf, errors="coerce").mean(), 2)})
        status["models"][m] = {"done": len(g), "complete": len(g) >= len(Q)}
    SUM = pd.DataFrame(rows); SUM.to_csv(f"{OUTPUT_DIR}/e1_summary.csv", index=False)
    pd.set_option("display.width", 250); print(SUM.T.to_string())
for m in MODELS:
    status["models"].setdefault(m, {"done": 0, "complete": False})
    sf = f"{OUTPUT_DIR}/results/e1/{m}.summary.json"
    if os.path.exists(sf): status["models"][m]["worker"] = json.load(open(sf))
status["all_complete"] = all(v["complete"] for v in status["models"].values())
json.dump(status, open(f"{OUTPUT_DIR}/e1_status.json", "w"), indent=2, default=str)
print("\nALL COMPLETE" if status["all_complete"] else "\nNOT COMPLETE -> upload this output as a dataset, set RESUME_ROOT to it and run again")

                                0         1        2        3
model                 InternVL-4B  Q25VL-7B  Q3VL-2B  Q3VL-4B
done                         4430      4430     4430     4430
complete                     True      True     True     True
parse_rate                    1.0       1.0      1.0      1.0
acc_3afc_main               0.448      0.38    0.266    0.284
acc_dstress>0                0.55     0.467    0.326    0.344
pair_acc_both_orders         0.55     0.467    0.326    0.344
p_answer_1                  0.564     0.287    0.692    0.252
p_answer_same                 0.0       0.0      0.0    0.008
acc_order_AB                0.594     0.267    0.523    0.094
acc_order_BA                0.506     0.666    0.129    0.594
acc_congruent               0.657     0.554    0.362    0.386
acc_neutral                 0.548     0.507    0.318    0.346
acc_conflict_nu             0.536     0.389    0.365    0.377
acc_conflict_eld            0.494     0.456    0.286    0.328
acc_conf

In [7]:
if len(R):
    ok = R[R.parse_ok & (R.set == "main")]
    fig, ax = plt.subplots(1, 3, figsize=(19, 4.3))
    acc = ok.groupby(["model", "dstress"]).correct.mean().unstack(0)
    acc.plot(ax=ax[0], marker="o"); ax[0].axhline(0.5, ls=":", c="grey"); ax[0].set_ylim(0, 1)
    ax[0].set_title("E1 accuracy vs Δstress (both orders; Δ=0 needs 'same')"); ax[0].set_xlabel("Δstress")
    nz = ok[ok.dstress > 0]
    nz.groupby(["cond", "model"]).correct.mean().unstack().reindex(["congruent", "neutral", "conflict_nu", "conflict_eld", "conflict_xr", "conflict_all"]).plot.bar(ax=ax[1])
    ax[1].axhline(0.5, ls=":", c="grey"); ax[1].set_ylim(0, 1); ax[1].set_title("E1 accuracy by condition (Δ>0)"); ax[1].set_xlabel("")
    e1b = R[R.parse_ok & (R.set == "e1b_mooney")]
    if len(e1b):
        e1b.groupby(["model", "dstress"]).correct.mean().unstack(0).plot(ax=ax[2], marker="s")
        ax[2].axhline(0.5, ls=":", c="grey"); ax[2].set_ylim(0, 1); ax[2].set_title("E1b Mooney-style pairs (degenerate, Δ 0.05-0.40)")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e1_overview.png", dpi=120); plt.show()

In [8]:
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N05 E1 · {status['hours']} h · all complete: {status['all_complete']}",
         " | ".join(f"{m}: {v['done']}/{len(Q)}" for m, v in status["models"].items()), ""]
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):45s} {os.path.getsize(p)/1024:9.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N05 E1 · 9.28 h · all complete: True
InternVL-4B: 4430/4430 | Q25VL-7B: 4430/4430 | Q3VL-2B: 4430/4430 | Q3VL-4B: 4430/4430

e1_queries.csv                                   1006.9 KB
e1_status.json                                      2.2 KB
e1_summary.csv                                      0.7 KB
fig_e1_overview.png                                99.6 KB
pdcp_infer.py                                      16.3 KB
logs/InternVL-4B.log                                7.8 KB
logs/Q25VL-7B.log                                  17.6 KB
logs/Q3VL-2B.log                                    6.5 KB
logs/Q3VL-4B.log                                    7.6 KB
results/e1/InternVL-4B.jsonl                     2883.3 KB
results/e1/InternVL-4B.summary.json                 0.3 KB
results/e1/Q25VL-7B.jsonl                        2628.2 KB
results/e1/Q25VL-7B.summary.json                    0.3 KB
results/e1/Q3VL-2B.jsonl                         4175.2 KB
results/e1/Q3VL-2B.summary.json                  